## Install Packages

In [ ]:
!pip install crewai crewai-tools openai python-dotenv yfinance duckduckgo_search pandas

In [1]:
import os
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, LLM
import yfinance as yf
from duckduckgo_search import DDGS
from crewai.tools import BaseTool

load_dotenv(override=True)
print("OPENAI_API_KEY loaded:", bool(os.getenv("OPENAI_API_KEY")))

OPENAI_API_KEY loaded: True


In [2]:
llm = LLM(
    model="openai/gpt-4.1-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.7
)

print(llm.call("Say hello in one short sentence."))

Hello!


In [3]:
class StockPriceTool(BaseTool):
    name: str = "Stock Price Fetcher"
    description: str = "Fetches 1 month stock price history for a given ticker symbol"

    def _run(self, ticker: str) -> str:
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(period="1mo")

            if hist.empty:
                return f"No stock data found for ticker: {ticker}"

            latest_close = hist["Close"].iloc[-1]
            highest_close = hist["Close"].max()
            lowest_close = hist["Close"].min()
            avg_close = hist["Close"].mean()

            return (
                f"Stock data for {ticker} (last 1 month):\n"
                f"- Latest Close: {latest_close:.2f}\n"
                f"- Highest Close: {highest_close:.2f}\n"
                f"- Lowest Close: {lowest_close:.2f}\n"
                f"- Average Close: {avg_close:.2f}\n"
            )
        except Exception as e:
            return f"Error fetching stock data for {ticker}: {str(e)}"


class StockNewsTool(BaseTool):
    name: str = "Stock News Searcher"
    description: str = "Searches recent stock-related news for a company or ticker"

    def _run(self, query: str) -> str:
        try:
            results_text = []
            with DDGS() as ddgs:
                results = ddgs.text(f"{query} stock news", max_results=5)
                for idx, item in enumerate(results, start=1):
                    title = item.get("title", "No title")
                    body = item.get("body", "No summary")
                    href = item.get("href", "No link")
                    results_text.append(
                        f"{idx}. Title: {title}\nSummary: {body}\nLink: {href}\n"
                    )

            if not results_text:
                return f"No recent news found for: {query}"

            return "\n".join(results_text)
        except Exception as e:
            return f"Error fetching news for {query}: {str(e)}"

In [4]:
price_tool = StockPriceTool()
news_tool = StockNewsTool()

stock_analyst = Agent(
    role="Stock Analyst",
    goal="Analyze recent stock price trends and latest market news for a given company",
    backstory=(
        "You are a highly skilled financial analyst. You study stock performance, "
        "market movement, and recent company-related news to produce practical insights."
    ),
    tools=[price_tool, news_tool],
    llm=llm,
    verbose=True
)

report_writer = Agent(
    role="Financial Report Writer",
    goal="Write a concise, investor-friendly stock report based on the analyst's findings",
    backstory=(
        "You are an expert financial content writer. You convert financial analysis "
        "into clear, structured, and professional reports for investors."
    ),
    llm=llm,
    verbose=True
)

In [5]:
search_task = Task(
    description=(
        "Analyze Apple Inc. (ticker: AAPL). "
        "Use the stock price tool to fetch 1 month of stock data. "
        "Use the news tool to gather recent relevant news about Apple stock. "
        "Summarize the current trend, notable price movement, and key recent news."
    ),
    expected_output=(
        "A structured financial analysis of Apple stock, including recent price trends, "
        "technical highlights, and recent news summary."
    ),
    agent=stock_analyst
)

analysis_task = Task(
    description=(
        "Using the stock analyst's findings, write a concise investor-friendly report "
        "for Apple Inc. (AAPL). Include:\n"
        "1. Market Summary\n"
        "2. Key Trends and Technical Highlights\n"
        "3. Recent News Impact\n"
        "4. Investment Outlook\n"
        "Keep it professional, readable, and practical."
    ),
    expected_output=(
        "A polished investor report with clear sections and practical commentary."
    ),
    agent=report_writer
)

crew = Crew(
    agents=[stock_analyst, report_writer],
    tasks=[search_task, analysis_task],
    verbose=True
)

In [6]:
result = crew.kickoff()
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c59c5db8-253b-448c-ab09-6a8e5c2548cd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze Apple Inc. (ticker: AAPL). Use the stock price tool to fetch 1 month of stock data. Use the      │
│  news tool to gather recent relevant news about Apple stock. Summarize the current trend, notable price         │
│  movement, and key recent news.                                                                                 │
│  ID: c01a5e13-d5c0-43e5-8db0-0d86d57f34e2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Analyst                                                                                           │
│                                                                                                                 │
│  Task: Analyze Apple Inc. (ticker: AAPL). Use the stock price tool to fetch 1 month of stock data. Use the      │
│  news tool to gather recent relevant news about Apple stock. Summarize the current trend, notable price         │
│  movement, and key recent news.                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: stock_price_fetcher                                                                                      │
│  Args: {'ticker': 'AAPL'}                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

C:\Users\RakeshKumarSingh\AppData\Local\Temp\ipykernel_33752\2537976710.py:36: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Financial Report Writer                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Apple Inc. (AAPL) Investor Report                                                                              │
│                                                                                                                 │
│  1. Market Summary                                                                                              │
│  Over the past month, Apple Inc.’s stock has exhibited relative stability, trading within a narrow range. The   │
│  highest closing price reached 260.81, while the lowest was 246.63. The latest closing price of 260.48 is       │
│  close to the monthly high, reflecting an upward momentum compared to the month’s average closing price of      │
│  253.74. This positioning indicates steady investor interest and a potential shift toward bullish sentiment.    │
│                                                                                                                 │
│  2. Key Trends and Technical Highlights                                                                         │
│  Apple’s stock has maintained a tight trading range between approximately 246.63 and 260.81, with no            │
│  significant volatility spikes. The price approaching the upper bound of this range suggests buyers are active  │
│  near these levels, reinforcing a steady and controlled market trend. The absence of erratic price movements    │
│  points to a balanced supply-demand dynamic, which may appeal to investors seeking stability.                   │
│                                                                                                                 │
│  3. Recent News Impact                                                                                          │
│  There have been no notable news developments related to Apple in the recent period. This lack of major         │
│  announcements implies that the stock’s price movements are primarily influenced by general market conditions   │
│  rather than company-specific catalysts. The current calm environment provides a stable backdrop for the        │
│  stock, although investors should remain attentive to any forthcoming news that could alter this status quo.    │
│                                                                                                                 │
│  4. Investment Outlook                                                                                          │
│  Apple’s stock demonstrates a stable and slightly bullish trend, currently trading near its monthly peak. The   │
│  steady trading range and absence of volatility suggest a low-risk profile in the near term. Investors may      │
│  consider maintaining or initiating positions while monitoring for upcoming earnings reports, product           │
│  launches, or market shifts that could impact price direction. Given the company’s strong fundamentals and      │
│  market position, Apple remains a compelling option for those seeking consistent performance within the         │
│  technology sector.                                                                                             │
│                                                                                                                 │
│  In summary, Apple Inc. offers a blend of stability and modest upward momentum, making it a practical           │
│  consideration for investors with a medium-term horizon

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the stock analyst's findings, write a concise investor-friendly report for Apple Inc. (AAPL).      │
│  Include:                                                                                                       │
│  1. Market Summary                                                                                              │
│  2. Key Trends and Technical Highlights                                                                         │
│  3. Recent News Impact                                                                                          │
│  4. Investment Outlook                                                                                          │
│  Keep it professional, readable, and practical.                                                                 │
│  Agent: Financial Report Writer                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: c59c5db8-253b-448c-ab09-6a8e5c2548cd                                                                       │
│  Final Output: Apple Inc. (AAPL) Investor Report                                                                │
│                                                                                                                 │
│  1. Market Summary                                                                                              │
│  Over the past month, Apple Inc.’s stock has exhibited relative stability, trading within a narrow range. The   │
│  highest closing price reached 260.81, while the lowest was 246.63. The latest closing price of 260.48 is       │
│  close to the monthly high, reflecting an upward momentum compared to the month’s average closing price of      │
│  253.74. This positioning indicates steady investor interest and a potential shift toward bullish sentiment.    │
│                                                                                                                 │
│  2. Key Trends and Technical Highlights                                                                         │
│  Apple’s stock has maintained a tight trading range between approximately 246.63 and 260.81, with no            │
│  significant volatility spikes. The price approaching the upper bound of this range suggests buyers are active  │
│  near these levels, reinforcing a steady and controlled market trend. The absence of erratic price movements    │
│  points to a balanced supply-demand dynamic, which may appeal to investors seeking stability.                   │
│                                                                                                                 │
│  3. Recent News Impact                                                                                          │
│  There have been no notable news developments related to Apple in the recent period. This lack of major         │
│  announcements implies that the stock’s price movements are primarily influenced by general market conditions   │
│  rather than company-specific catalysts. The current calm environment provides a stable backdrop for the        │
│  stock, although investors should remain attentive to any forthcoming news that could alter this status quo.    │
│                                                                                                                 │
│  4. Investment Outlook                                                                                          │
│  Apple’s stock demonstrates a stable and slightly bullish trend, currently trading near its monthly peak. The   │
│  steady trading range and absence of volatility suggest a low-risk profile in the near term. Investors may      │
│  consider maintaining or initiating positions while monitoring for upcoming earnings reports, product           │
│  launches, or market shifts that could impact price direction. Given the company’s strong fundamentals and      │
│  market position, Apple remains a compelling option for those seeking consistent performance within the         │
│  technology sector.                                                                                             │
│                                                                                                                 │
│  In summary, Apple Inc. offers a blend of stability and modest upward momentum, making it a practical           │
│  consideration for investors with a medium-term horizo

Apple Inc. (AAPL) Investor Report

1. Market Summary  
Over the past month, Apple Inc.’s stock has exhibited relative stability, trading within a narrow range. The highest closing price reached 260.81, while the lowest was 246.63. The latest closing price of 260.48 is close to the monthly high, reflecting an upward momentum compared to the month’s average closing price of 253.74. This positioning indicates steady investor interest and a potential shift toward bullish sentiment.

2. Key Trends and Technical Highlights  
Apple’s stock has maintained a tight trading range between approximately 246.63 and 260.81, with no significant volatility spikes. The price approaching the upper bound of this range suggests buyers are active near these levels, reinforcing a steady and controlled market trend. The absence of erratic price movements points to a balanced supply-demand dynamic, which may appeal to investors seeking stability.

3. Recent News Impact  
There have been no notable news develop

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯